# TraceWhisperer 입문 튜토리얼
## Arm 디버그 트레이스와 전력 분석의 결합

---

## 이 노트북에 대하여

이 튜토리얼은 **ChipWhisperer Husky**와 **STM32F3** 타겟을 사용하여  
**TraceWhisperer**를 처음 접하는 분들을 위한 단계별 입문 가이드입니다.

ChipWhisperer로 전력 분석(Power Analysis)을 해본 경험이 있다면,  
TraceWhisperer는 여기에 **Arm 프로세서의 디버그 트레이스** 기능을 더해줍니다.  
전력 파형만으로는 알기 어려웠던 *'어느 코드가 실행되고 있는가'*를  
정확히 파악할 수 있게 됩니다.

### 핵심 아이디어

```
기존 ChipWhisperer:   전력 파형만 캡처
                      → "지금 뭔가 하고 있는데... 뭐지?"

TraceWhisperer 추가:  전력 파형 + Arm 트레이스 이벤트 동시 캡처
                      → "SubBytes() 함수가 지금 이 시점에 실행됐다!"
```

### 요구 하드웨어

| 항목 | 사양 |
|------|------|
| 캡처 보드 | ChipWhisperer Husky (또는 Husky Plus) |
| 타겟 보드 | CW308 + STM32F3 |
| 점퍼 케이블 | 최소 3개 (TMS/TCK/TDO 연결용) |
| 펌웨어 | simpleserial-trace (STM32F3 빌드) |

### 이 노트북의 학습 목표

1. TraceWhisperer의 SWO 트레이스 모드 설정
2. AES 연산 중 특정 함수(SubBytes, AddRoundKey)의 실행 시점 포착
3. 전력 파형 위에 트레이스 이벤트를 오버레이하여 시각화
4. 트레이스 기반 분석의 활용 가능성 이해

---

> **⚠️ 주의**: 이 노트북은 `simpleserial-trace` 전용 펌웨어가 필요합니다.  
> 일반 `simpleserial-aes` 펌웨어와는 다릅니다!

---
## 사전 지식: TraceWhisperer란?

### Arm 프로세서의 디버그 트레이스 기능

Arm Cortex-M 계열 프로세서에는 실시간으로 디버그 정보를 내보내는  
하드웨어가 내장되어 있습니다. 이를 **CoreSight** 디버그 아키텍처라고 합니다.

우리가 이 튜토리얼에서 사용할 핵심 구성요소:

| 구성요소 | 역할 | 이 튜토리얼에서의 사용 |
|----------|------|------------------------|
| **DWT** (Data Watchpoint and Trace) | PC 값 비교 및 이벤트 생성 | 특정 함수 주소와 PC가 일치할 때 이벤트 발생 |
| **ETM** (Embedded Trace Macrocell) | 명령어 추적 패킷 생성 | PC 일치 이벤트 → 트레이스 패킷으로 출력 |
| **TPI/SWO** (Trace Port Interface) | 트레이스 데이터 직렬 출력 | SWO 핀 1개로 모든 트레이스 데이터 전송 |

### 트레이스 인터페이스: SWO vs 병렬 트레이스

트레이스 데이터를 내보내는 방법은 두 가지입니다:

- **SWO (Serial Wire Output)**: 핀 1개, 낮은 대역폭, 우리 타겟에서 지원 ✅
- **병렬 트레이스**: 핀 4~5개, 높은 대역폭, 특별한 타겟 필요

이 튜토리얼에서는 STM32F3이 지원하는 **SWO 모드**를 사용합니다.

### TraceWhisperer의 핵심 기능: 패턴 매칭

SWO 트레이스 데이터는 복잡한 바이너리 형식(TPIU 포맷)으로 나옵니다.  
이를 소프트웨어로 파싱하면 지연이 생겨 실시간 트리거로 쓰기 어렵습니다.

TraceWhisperer는 **FPGA 내부에서 패턴 매칭**을 수행합니다:
- 사전에 정의한 바이트 패턴이 트레이스 스트림에 나타나는 순간을 감지
- 해당 시점의 **타임스탬프**만 기록 → 매우 간단하고 정확함
- 소프트웨어 파싱 없이 하드웨어 레벨에서 처리

---
## Step 0: 하드웨어 연결 확인

코드 실행 전에 하드웨어 연결을 반드시 확인하세요.

### SWO 트레이스를 위한 점퍼 케이블 연결

Arm SWD/JTAG 인터페이스의 핀을 Husky의 USERIO 포트에 연결합니다.

```
STM32F3 타겟 (CW308)         Husky USERIO 포트
─────────────────────         ─────────────────
TMS  (JTAG 모드 선택)   ────►  D0
TCK  (JTAG 클럭)        ────►  D1  
TDO  (트레이스 출력)    ────►  D2
```

CW308 보드에서 이 핀들의 위치:
- **TMS**: 20핀 커넥터의 핀 1 (또는 보드의 JTAG 헤더)
- **TCK**: 20핀 커넥터의 핀 9
- **TDO (SWO)**: 20핀 커넥터의 핀 3

> **💡 팁**: Husky와 CW308은 20핀 UFO 커넥터로 이미 연결되어 있습니다.  
> USERIO 포트만 별도 점퍼 케이블로 연결해주면 됩니다.

### 연결 확인 체크리스트
- [ ] CW308과 Husky가 20핀 커넥터로 연결됨
- [ ] TMS → D0 연결됨
- [ ] TCK → D1 연결됨  
- [ ] TDO → D2 연결됨
- [ ] STM32F3에 `simpleserial-trace` 펌웨어가 준비됨

---
## Step 1: 환경 설정 및 기본 임포트

In [ ]:
# ─────────────────────────────────────────────────────────────
# 기본 라이브러리 임포트
# ─────────────────────────────────────────────────────────────
import chipwhisperer as cw
import numpy as np
import time

print(f"ChipWhisperer 버전: {cw.__version__}")
print("✅ 라이브러리 임포트 성공")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 플랫폼 설정
#
# 이 튜토리얼은 Husky + STM32F3 조합에 최적화되어 있습니다.
# 다른 조합을 사용한다면 아래 변수들을 수정하세요.
# ─────────────────────────────────────────────────────────────

PLATFORM        = 'CW308_STM32F3'   # 타겟 MCU (STM32F3)
SCOPETYPE       = 'OPENADC'         # Husky 사용시 OPENADC
TRACE_PLATFORM  = 'Husky'          # TraceWhisperer 플랫폼
TRACE_INTERFACE = 'swo'            # 트레이스 인터페이스 (SWO 또는 parallel)

print(f"타겟 플랫폼  : {PLATFORM}")
print(f"캡처 보드   : {TRACE_PLATFORM}")
print(f"트레이스 방식: {TRACE_INTERFACE.upper()}")

In [ ]:
%%bash -s "$PLATFORM"

cd simpleserial-trace/
make PLATFORM=$1 clean
make PLATFORM=$1

f=simpleserial-trace-CW308_STM32F3.elf

arm-none-eabi-objdump -D -x -s -S -l "$f" > "${f%.elf}.txt"

---
## Step 2: Husky 연결 및 초기화

ChipWhisperer Husky에 연결하고, 기본 측정 파라미터를 설정합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Husky 연결
#
# scope: 파형 캡처 장치 (Husky 본체)
# target: 통신 대상 (STM32F3)
# ─────────────────────────────────────────────────────────────

try:
    scope = cw.scope()
    target = cw.target(scope, cw.targets.SimpleSerial)
    print("✅ Husky 연결 성공")
except Exception as e:
    print(f"❌ 연결 실패: {e}")
    print("   → USB 케이블 연결 상태를 확인하세요.")
    raise

In [ ]:
# ─────────────────────────────────────────────────────────────
# 기본 파라미터 설정
#
# 아래 설정들은 일반적인 AES 캡처에 최적화된 값입니다.
# ─────────────────────────────────────────────────────────────

# 클럭 설정
scope.clock.clkgen_freq = 7.37e6  # 타겟 클럭: 7.37 MHz

# ADC 샘플 수: AES 한 라운드 전체를 캡처할 수 있는 충분한 길이
# Husky는 최대 adc_mul = 10배 오버샘플링 지원
scope.adc.samples = 31000

# 전압 게인: STM32F3 타겟에 맞는 값
scope.gain.db = 12

# I/O 설정
scope.io.tio1 = 'serial_rx'
scope.io.tio2 = 'serial_tx'
scope.io.hs2  = 'clkgen'

# 트리거 설정
scope.trigger.triggers = 'tio4'   # TIO4 핀의 HIGH 신호로 트리거

print("✅ 기본 파라미터 설정 완료")
print(f"   클럭 주파수 : {scope.clock.clkgen_freq/1e6:.2f} MHz")
print(f"   ADC 샘플 수 : {scope.adc.samples:,} 샘플")
print(f"   전압 게인   : {scope.gain.db} dB")

---
## Step 3: 타겟 펌웨어 프로그래밍

TraceWhisperer 실습에는 **simpleserial-trace** 전용 펌웨어가 필요합니다.  
이 펌웨어는 일반 simpleserial-aes와 기능이 같지만,  
STM32의 SWO 트레이스 핀을 초기화하는 코드가 추가되어 있습니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 펌웨어 확인 및 프로그래밍
#
# 이미 올바른 펌웨어가 올라가 있으면 재프로그래밍을 건너뜁니다.
# 최초 실행 시 또는 다른 펌웨어가 올라가 있을 때만 프로그래밍합니다.
# ─────────────────────────────────────────────────────────────

# Husky 내장 TraceWhisperer 활성화 (연결 및 기본 초기화)
scope.trace.target = target
scope.trace.enabled = True

# 현재 펌웨어 버전 확인
fw_buildtime = scope.trace.get_fw_buildtime()
print(f"현재 펌웨어: {fw_buildtime}")

# simpleserial-trace 펌웨어 경로 (실제 경로에 맞게 수정하세요)
FW_PATH = 'simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex'

# 펌웨어 프로그래밍
print("\n⚠️  simpleserial-trace 펌웨어를 프로그래밍합니다...")
try:
    prog = cw.programmers.STM32FProgrammer
    cw.program_target(scope, prog, FW_PATH)
    print("✅ 펌웨어 프로그래밍 완료")
except FileNotFoundError:
    print("❌ 펌웨어 파일을 찾을 수 없습니다.")
    print(f"   경로를 확인하세요: {FW_PATH}")
    raise

In [ ]:
# ─────────────────────────────────────────────────────────────
# 타겟 리셋 함수 정의 및 실행
#
# 펌웨어 프로그래밍 후 또는 타겟이 응답 없을 때 리셋이 필요합니다.
# ─────────────────────────────────────────────────────────────

def reset_target(scope, delay=0.05):
    """타겟 MCU를 리셋합니다. nRST 핀을 LOW로 내렸다가 HIGH로 올립니다."""
    scope.io.nrst = 'low'
    time.sleep(delay)
    scope.io.nrst = 'high'
    time.sleep(delay)

reset_target(scope)
print("✅ 타겟 리셋 완료")

# 타겟이 살아있는지 확인
fw_buildtime = scope.trace.get_fw_buildtime()
if fw_buildtime:
    print(f"✅ 타겟 응답 확인: {fw_buildtime}")
else:
    print("❌ 타겟이 응답하지 않습니다. 연결 상태를 확인하세요.")

---
## Step 4: SWO 트레이스 인터페이스 설정

이 단계가 TraceWhisperer 설정의 핵심입니다.  
Arm 프로세서는 기본적으로 **JTAG 모드**로 시작하는데,  
SWO 트레이스를 사용하려면 **SWD 모드**로 전환해야 합니다.

### SWO 클럭 설정의 이해

SWO 데이터의 전송 속도는 두 가지 파라미터로 결정됩니다:

```
SWO 비트 주기 = (TPI.ACPR + 1) × 타겟 클럭 주기

TraceWhisperer 복원 클럭 = 타겟 클럭 × trigger_freq_mul
swo_div = trigger_freq_mul × (ACPR + 1)
```

- `ACPR = 0`: 가장 빠른 SWO 속도 (타겟 클럭과 동일한 속도)
- `trigger_freq_mul = 8`: TraceWhisperer가 SWO 클럭의 8배 속도로 샘플링

In [ ]:
# ─────────────────────────────────────────────────────────────
# SWO 트레이스 클럭 설정
# ─────────────────────────────────────────────────────────────

# 타겟 클럭을 TraceWhisperer의 기준 클럭으로 사용합니다.
# 이렇게 하면 트레이스 타임스탬프와 전력 파형의 시간 기준이 같아집니다.
scope.trace.clock.fe_clock_src = 'target_clock'

# 클럭이 정상적으로 감지되는지 확인
if not scope.trace.clock.fe_clock_alive:
    raise RuntimeError(
        "❌ 타겟 클럭이 감지되지 않습니다!\n"
        "   → HS2 핀과 타겟의 클럭 출력이 연결되어 있는지 확인하세요."
    )

print(f"✅ 클럭 감지 확인: {scope.trace.clock.fe_clock_src}")
print(f"   현재 클럭 주파수: {scope.clock.clkgen_freq/1e6:.2f} MHz")

In [ ]:
# ─────────────────────────────────────────────────────────────
# JTAG → SWD 모드 전환
#
# Arm 프로세서는 리셋 후 JTAG 모드로 시작합니다.
# SWO 트레이스를 사용하려면 SWD 모드로 전환해야 합니다.
#
# jtag_to_swd()는 TMS/TCK 핀에 특수한 시퀀스를 전송하여
# 프로세서를 SWD 모드로 전환합니다.
# ─────────────────────────────────────────────────────────────

# SWO 모드 활성화
scope.trace.trace_mode = 'SWO'

# JTAG → SWD 전환 (TMS/TCK 핀에 특수 시퀀스 전송)
scope.trace.jtag_to_swd()
print("✅ JTAG → SWD 모드 전환 완료")

In [ ]:
# ─────────────────────────────────────────────────────────────
# SWO 클럭 파라미터 설정
# ─────────────────────────────────────────────────────────────

acpr = 0           # TPI.ACPR: SWO 속도 조절 레지스터 (0 = 최고 속도)
trigger_freq_mul = 8  # TraceWhisperer 오버샘플링 배율

# TraceWhisperer의 SWO 복원 클럭 주파수 설정
# = 타겟 클럭의 8배 (오버샘플링하여 안정적인 SWO 복원)
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * trigger_freq_mul

# 타겟의 TPI.ACPR 레지스터에 값 쓰기 (SWO 출력 속도 설정)
scope.trace.target_registers.TPI_ACPR = acpr

# FPGA에 SWO 비트당 클럭 수 알려주기
scope.trace.swo_div = trigger_freq_mul * (acpr + 1)

# SWO 클럭 PLL이 잠겼는지 (locked) 확인
if not scope.trace.clock.swo_clock_locked:
    raise RuntimeError(
        "❌ SWO 클럭 PLL이 잠기지 않았습니다!\n"
        "   → 클럭 설정을 다시 확인하세요."
    )

print(f"✅ SWO 클럭 설정 완료")
print(f"   ACPR        : {acpr}")
print(f"   오버샘플링  : {trigger_freq_mul}배")
print(f"   SWO 클럭    : {scope.trace.clock.swo_clock_freq/1e6:.2f} MHz")
print(f"   swo_div     : {scope.trace.swo_div}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# SWO 라인 상태 확인
#
# SWD 모드가 정상적으로 활성화되면 SWO 라인이 HIGH를 유지합니다.
# Husky에서는 USERIO D2 핀의 상태로 확인합니다.
# ─────────────────────────────────────────────────────────────

# USERIO 상태 레지스터의 비트 2가 SWO 라인 (D2 핀)
swo_line_high = scope.userio.status & 0x4

if swo_line_high:
    print("✅ SWO 라인 정상 (HIGH)")
    print("   SWD 모드가 정상적으로 활성화되었습니다.")
else:
    print("⚠️  SWO 라인이 LOW입니다.")
    print("   점퍼 케이블 연결 및 타겟 상태를 확인하세요.")
    print("   타겟 리셋 후 다시 시도해보세요: reset_target(scope)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# SWD 모드 전환 후 타겟 상태 재확인
#
# jtag_to_swd() 후 일부 타겟이 응답 불능이 될 수 있습니다.
# 이 경우 리셋이 필요합니다.
# ─────────────────────────────────────────────────────────────

fw_buildtime = scope.trace.get_fw_buildtime()

if not fw_buildtime:
    print("타겟이 응답하지 않습니다. 리셋을 시도합니다...")
    reset_target(scope)
    time.sleep(0.1)
    fw_buildtime = scope.trace.get_fw_buildtime()

if fw_buildtime:
    print(f"✅ 타겟 정상 작동: {fw_buildtime}")
else:
    print("❌ 타겟이 여전히 응답하지 않습니다.")
    print("   하드웨어 연결을 다시 확인하세요.")

---
## Step 5: 트레이스 캡처 모드 설정

TraceWhisperer에는 두 가지 캡처 모드가 있습니다:

| 모드 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **Raw 캡처** | 모든 트레이스 데이터를 원시 형태로 저장 | 상세 분석 가능 | 파싱 도구 필요, 복잡 |
| **패턴 매칭** | 특정 패턴과 일치하는 순간의 타임스탬프만 저장 | 간단, 실시간 처리 가능 | 사전 정의된 패턴만 감지 |

이 튜토리얼에서는 **패턴 매칭 모드**를 사용합니다.  
DWT_COMP 이벤트 패킷은 항상 `[3, 8, 32]` 바이트로 시작하기 때문에  
이를 패턴으로 등록하면 PC 일치 이벤트를 정확히 감지할 수 있습니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SWO 싱크 프레임 비활성화
#
# 기본적으로 Arm 프로세서는 16,000,000 클럭마다 '싱크 프레임'을
# 주기적으로 전송합니다. 처음 디버깅 시 연결 확인에 유용하지만,
# 실제 캡처에서는 이 프레임이 타이밍을 방해할 수 있습니다.
#
# DWT_CTRL 레지스터 설정: 주기적 PC 샘플링은 끄되, 비교 이벤트는 활성화
# 0x40000021 = CYCCNTENA(비트0) + EXCTRCENA(비트16) + NOPROFTRAP(비트30)
# ─────────────────────────────────────────────────────────────

scope.trace.target_registers.DWT_CTRL = 0x40000021
print("✅ SWO 싱크 프레임 비활성화 완료")
print("   (캡처 중 싱크 프레임이 타이밍을 방해하지 않습니다)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 트리거 소스 설정
#
# TraceWhisperer의 캡처 시작을 언제 결정하는지 설정합니다.
#
# 옵션 1: 'firmware trigger' - 타겟 펌웨어의 TIO4 핀 신호로 트리거
#          → 안정적, 캡처 시작점이 명확함 (이 튜토리얼에서 사용)
#
# 옵션 2: 0 (패턴 매칭 룰 번호) - 트레이스 패턴이 나타나면 트리거
#          → 최대 6 클럭 지터 발생 가능, 고급 사용자용
# ─────────────────────────────────────────────────────────────

scope.trace.capture.trigger_source = 'firmware trigger'
print("✅ 트리거 소스: 펌웨어 트리거 (TIO4 핀)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 패턴 매칭 모드 설정
#
# DWT_COMP 이벤트 패킷의 헤더 바이트: [3, 8, 32]
# - 3  = SWIT 패킷 소스 ID
# - 8  = 비교 이벤트 타입
# - 32 = PC 일치 이벤트 코드
#
# 이 3바이트 패턴이 SWO 스트림에 나타날 때마다 타임스탬프를 기록합니다.
# ─────────────────────────────────────────────────────────────

# Raw 캡처 비활성화 → 패턴 매칭 모드 사용
scope.trace.capture.raw = False

# 패턴 매칭 룰 0번에 DWT_COMP 이벤트 패턴 등록
scope.trace.set_pattern_match(0, [3, 8, 32])

# 룰 0번 활성화
scope.trace.capture.rules_enabled = [0]

print("✅ 패턴 매칭 설정 완료")
print("   감지 패턴: [0x03, 0x08, 0x20] (DWT_COMP PC 일치 이벤트)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 캡처 지속 시간 설정
#
# 'while_trig': 타겟 트리거 핀이 HIGH인 동안만 캡처
#              → 가장 직관적이며 전력 캡처와 타이밍이 일치함
#
# 'count_writes': 지정한 이벤트 수만큼 캡처 후 종료
# ─────────────────────────────────────────────────────────────

scope.trace.capture.mode = 'while_trig'
print("✅ 캡처 모드: while_trig (트리거 HIGH 동안 지속 캡처)")

---
## Step 6: PC 주소 매칭 설정

DWT (Data Watchpoint and Trace) 유닛의 비교기(COMP)에  
감시할 함수의 시작 주소를 등록합니다.

### 주소를 어떻게 찾는가?

컴파일된 펌웨어의 ELF 파일을 역어셈블하면 각 함수의 주소를 알 수 있습니다:

```bash
arm-none-eabi-objdump -d simpleserial-trace-CW308_STM32F3.elf | grep -E "<SubBytes>|<AddRoundKey>"
```

아래 주소는 **사전 컴파일된 simpleserial-trace** 펌웨어 기준입니다.  
직접 컴파일한 경우 최적화 옵션에 따라 주소가 달라질 수 있습니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# AES 함수 주소 설정 (simpleserial-trace STM32F3 사전 빌드 기준)
#
# SubBytes()   : AES 비선형 치환 단계 (S-Box 적용)
# AddRoundKey(): 라운드 키와 XOR 연산
#
# 이 두 함수는 AES의 핵심 연산으로, 전력 분석 공격의 주요 타겟입니다.
# ─────────────────────────────────────────────────────────────
# 매칭할 PC(Program Counter) 주소 설정
# 이 주소는 simpleserial-trace 펌웨어 기준입니다. (chipwhiserer 공식 깃허브의 hex 파일 기준)
# 0x080018c4: SubBytes() 함수의 시작 주소    -> 새 빌드 후 simpleserial-trace-CW308_STM32F3.txt로 확인한 주소  0x080017d8
# 0x0800188c: AddRoundKey() 함수의 시작 주소 -> 새 빌드 후 simpleserial-trace-CW308_STM32F3.txt로 확인한 주소  0x080017a4

ADDR_SUBBYTES    = 0x080017d8  # SubBytes() 함수 시작 주소
ADDR_ADDROUNDKEY = 0x080017a4  # AddRoundKey() 함수 시작 주소

# DWT.COMP0, DWT.COMP1 레지스터에 주소 등록
# ETM.TEEVR 레지스터에 이벤트 비교 설정
scope.trace.set_isync_matches(
    addr0=ADDR_SUBBYTES,
    addr1=ADDR_ADDROUNDKEY,
    match='both'    # 두 주소 모두 이벤트 발생
)

print("✅ PC 주소 매칭 설정 완료")
print(f"   DWT_COMP0 (SubBytes)   : 0x{ADDR_SUBBYTES:08X}")
print(f"   DWT_COMP1 (AddRoundKey): 0x{ADDR_ADDROUNDKEY:08X}")
print("   match='both': 두 주소 모두에서 트레이스 이벤트 발생")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 주기적 PC 샘플링 비활성화
#
# DWT는 두 가지 방식으로 PC 정보를 출력할 수 있습니다:
#   1. 주기적 샘플링: 일정 간격마다 현재 PC 값을 출력
#   2. 비교 이벤트: COMP 레지스터와 일치할 때만 출력 (우리가 사용하는 방식)
#
# 두 가지가 동시에 활성화되면 SWO 대역폭을 낭비하고 이벤트가 섞일 수 있습니다.
# ─────────────────────────────────────────────────────────────

scope.trace.set_periodic_pc_sampling(enable=0)
print("✅ 주기적 PC 샘플링 비활성화")
print("   (DWT_COMP 이벤트만 SWO로 출력됩니다)")

---
## Step 7: 첫 번째 캡처 실행

모든 설정이 완료되었습니다. 이제 실제 캡처를 실행해봅니다.

캡처 순서:
1. `scope.trace.arm_trace()` → TraceWhisperer 캡처 준비
2. `cw.capture_trace()` → 전력 파형 캡처 (동시에 트레이스 캡처도 실행됨)
3. `scope.trace.read_capture_data()` → 트레이스 데이터 읽기
4. `get_rule_match_times()` → 패턴 일치 타임스탬프 추출

In [ ]:
# ─────────────────────────────────────────────────────────────
# 단일 캡처 실행
# ─────────────────────────────────────────────────────────────

# 1단계: TraceWhisperer를 캡처 대기 상태로 전환
scope.trace.arm_trace()

# 2단계: 전력 파형 + 트레이스 동시 캡처
# AES 암호화: 16바이트 평문 (0x00*16), 16바이트 키 (0x00*16)
powertrace = cw.capture_trace(
    scope,
    target,
    plaintext=bytearray(16),   # 평문: 16바이트 모두 0
    key=bytearray(16)          # 키: 16바이트 모두 0
)

# 캡처 성공 여부 확인
if powertrace is None:
    print("❌ 전력 파형 캡처 실패!")
    print("   타겟 리셋 후 다시 시도하세요: reset_target(scope)")
else:
    print("✅ 전력 파형 캡처 성공")
    print(f"   파형 길이: {len(powertrace.wave):,} 샘플")
    print(f"   평문: {powertrace.textin.hex()}")
    print(f"   암호문: {powertrace.textout.hex()}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 트레이스 데이터 읽기 및 파싱
# ─────────────────────────────────────────────────────────────

# TraceWhisperer FIFO가 비어있지 않은지 확인
if scope.trace.fifo_empty():
    print("⚠️  트레이스 FIFO가 비어있습니다.")
    print("   TraceWhisperer가 아무 이벤트도 감지하지 못했습니다.")
    print("   PC 주소 설정 및 점퍼 케이블 연결을 확인하세요.")
else:
    # 원시 트레이스 데이터 읽기
    raw = scope.trace.read_capture_data()

    # 패턴 매칭 이벤트의 타임스탬프 추출
    # verbose=True: 감지된 이벤트 정보를 출력
    # rawtimes=False: 타겟 클럭 기준 타임스탬프 (ADC 샘플 단위)
    times = scope.trace.get_rule_match_times(raw, rawtimes=False, verbose=True)

    print(f"\n✅ 트레이스 이벤트 총 {len(times)}개 감지")
    print("\n감지된 이벤트 타임스탬프 (단위: 타겟 클럭 사이클):")
    for i, t in enumerate(times):
        print(f"   이벤트 {i+1:2d}: {t[0]:8,} 사이클")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 캡처 통계 확인
#
# AES-128은 10라운드를 수행합니다.
# SubBytes()와 AddRoundKey()는 각 라운드에서 한 번씩 호출되지만,
# 마지막 라운드는 다른 코드 경로를 사용하기 때문에
# 일반적으로 21개의 이벤트가 감지됩니다 (10라운드 × 2함수 + 1함수(AddRoundKey)).
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("패턴 매칭 통계")
print("=" * 50)

# 마지막으로 매칭된 패턴 데이터
#
# ChipWhisperer 버전에 따라 matched_pattern_data의 반환 타입이 다를 수 있습니다.
#   - 구버전: 바이트 리스트 [3, 8, 32, ...]
#   - 신버전: 16진수 문자열 '03 08 20 ...'
# 아래 코드는 두 경우 모두 처리합니다.
matched_data = scope.trace.capture.matched_pattern_data

if isinstance(matched_data, str):
    # 문자열인 경우: 공백으로 분리한 뒤 정수로 변환
    try:
        matched_bytes = [int(x, 16) for x in matched_data.split() if x]
        print(f"마지막 매칭 데이터: {[hex(b) for b in matched_bytes]}")
    except ValueError:
        # 파싱 불가 시 원본 문자열 그대로 출력
        print(f"마지막 매칭 데이터 (raw): {matched_data!r}")
elif matched_data is None:
    print("마지막 매칭 데이터: 없음 (캡처 전 또는 이벤트 없음)")
else:
    # 바이트/리스트인 경우
    print(f"마지막 매칭 데이터: {[hex(b) for b in matched_data]}")

# 각 룰별 매칭 횟수
matched_counts = scope.trace.capture.matched_pattern_counts
print(f"룰 0 매칭 횟수   : {matched_counts[0]}회")
print()
print("💡 참고: AES-128 10라운드에서 약 21개 이벤트가 예상됩니다")
print("   (마지막 라운드는 MixColumns 없이 다른 경로로 실행)")

---
## Step 8: 전력 파형과 트레이스 이벤트 시각화

이제 가장 흥미로운 부분입니다.  
전력 파형 위에 트레이스 이벤트 시점을 수직선으로 표시합니다.

### 타임스탬프 스케일 변환

전력 파형과 트레이스 타임스탬프의 시간 기준이 다를 수 있습니다:

```
트레이스 타임스탬프: 타겟 클럭 단위 (7.37 MHz)
전력 파형 x축: ADC 샘플 단위 (타겟 클럭 × adc_mul 배)

따라서: 파형 x축 위치 = 트레이스 타임스탬프 × adc_mul
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# 스케일 변환 계수 계산
#
# Husky는 타겟 클럭의 adc_mul 배로 오버샘플링합니다.
# 기본값은 4배이지만 설정에 따라 달라질 수 있습니다.
# ─────────────────────────────────────────────────────────────

multiplier = scope.clock.adc_mul

print(f"ADC 오버샘플링 배율: {multiplier}×")
print(f"타겟 클럭: {scope.clock.clkgen_freq/1e6:.2f} MHz")
print(f"ADC 샘플링 속도: {scope.clock.clkgen_freq * multiplier / 1e6:.2f} MHz")
print()
print("트레이스 타임스탬프 → 파형 x축 변환:")
if times:
    for i, t in enumerate(times[:5]):
        print(f"  이벤트 {i+1}: {t[0]:6,} 사이클 → 파형 x = {t[0] * multiplier:8,}")
    if len(times) > 5:
        print(f"  ... (총 {len(times)}개)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bokeh를 이용한 인터랙티브 시각화
#
# - 빨간 선: 전력 파형 (AES 암호화 연산 중 측정)
# - 검은 수직선: 트레이스 이벤트 시점 (SubBytes 또는 AddRoundKey 호출)
#
# Bokeh 그래프는 마우스로 확대/축소 및 이동이 가능합니다!
# ─────────────────────────────────────────────────────────────

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span, Legend, LegendItem
from bokeh.resources import INLINE

# Jupyter 노트북 인라인 출력 설정
output_notebook(INLINE)

# 그래프 생성
p = figure(
    width=1200,
    height=400,
    title="전력 파형 + TraceWhisperer 이벤트 오버레이",
    x_axis_label="ADC 샘플",
    y_axis_label="전력 (정규화)",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# 전력 파형 그리기 (빨간 선)
x_range = list(range(len(powertrace.wave)))
power_line = p.line(
    x_range,
    powertrace.wave,
    line_color="red",
    line_width=1,
    alpha=0.8,
    legend_label="전력 파형"
)

# 트레이스 이벤트 수직선 그리기 (검은 선)
vlines = []
for t in times:
    x_pos = t[0] * multiplier  # 스케일 변환 적용
    vline = Span(
        location=x_pos,
        dimension='height',
        line_color='black',
        line_width=1.5,
        line_alpha=0.7
    )
    vlines.append(vline)

p.renderers.extend(vlines)

# 범례 추가
p.legend.location = "top_right"

show(p)
print(f"\n총 {len(times)}개의 트레이스 이벤트가 파형에 표시되었습니다.")

---
## Step 9: 다중 캡처로 통계적 분석

실제 측면채널 분석에서는 다수의 트레이스를 수집합니다.  
여기서는 여러 번 캡처하여 트레이스 이벤트 타이밍의 일관성을 확인합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 다중 캡처 수행
# ─────────────────────────────────────────────────────────────

from tqdm.notebook import trange

NUM_TRACES = 10  # 캡처 횟수 (실습용으로 10개, 실제 분석에는 수천 개 필요)

# 키 텍스트 쌍 생성기
ktp = cw.ktp.Basic()

# 결과 저장 리스트
powertraces_list = []    # 전력 파형
trace_times_list = []    # 트레이스 이벤트 타임스탬프
plaintexts = []
keys = []

print(f"{NUM_TRACES}개 트레이스 캡처 시작...\n")

for i in trange(NUM_TRACES, desc="캡처 진행"):
    # 랜덤 키/평문 생성
    key, text = ktp.next()

    # TraceWhisperer 캡처 준비
    scope.trace.arm_trace()

    # 전력 파형 캡처
    pt = cw.capture_trace(scope, target, text, key)

    if pt is None:
        print(f"⚠️  {i}번 캡처 실패, 건너뜀")
        continue

    # 트레이스 데이터 읽기 및 파싱
    if not scope.trace.fifo_empty():
        raw = scope.trace.read_capture_data()
        times_i = scope.trace.get_rule_match_times(raw, rawtimes=False, verbose=False)
    else:
        times_i = []

    powertraces_list.append(pt)
    trace_times_list.append(times_i)
    plaintexts.append(pt.textin)
    keys.append(pt.key)

print(f"\n✅ 캡처 완료: {len(powertraces_list)}/{NUM_TRACES}개 성공")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 이벤트 타이밍 일관성 분석
#
# AES는 결정론적 알고리즘이므로, 같은 라운드의 이벤트는
# 항상 동일한 시점에 발생해야 합니다.
# 타이밍 변동이 크다면 설정에 문제가 있을 수 있습니다.
# ─────────────────────────────────────────────────────────────

# 각 캡처의 이벤트 수 확인
event_counts = [len(t) for t in trace_times_list]

print("=" * 50)
print("이벤트 타이밍 통계")
print("=" * 50)
print(f"평균 이벤트 수: {np.mean(event_counts):.1f}")
print(f"최소 / 최대   : {min(event_counts)} / {max(event_counts)}")

# 이벤트 수가 일정한 캡처만 선택 (분석용)
expected_count = 21  # AES-128 예상 이벤트 수
valid_indices = [i for i, c in enumerate(event_counts) if c == expected_count]
print(f"\n유효 캡처 수 (이벤트 {expected_count}개): {len(valid_indices)}/{len(powertraces_list)}")

# 첫 번째 이벤트 타이밍 변동 분석
if valid_indices:
    first_event_times = [trace_times_list[i][0][0] for i in valid_indices]
    print(f"\n첫 이벤트 타이밍 통계 (타겟 클럭 사이클):")
    print(f"  평균: {np.mean(first_event_times):,.1f}")
    print(f"  표준편차: {np.std(first_event_times):.2f}")
    print(f"  범위: {min(first_event_times):,} ~ {max(first_event_times):,}")
    print()
    if np.std(first_event_times) < 2:
        print("✅ 이벤트 타이밍이 매우 일관됩니다 (표준편차 < 2 사이클)")
    else:
        print("⚠️  타이밍 변동이 있습니다. SWO 설정을 확인하세요.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 여러 파형 오버레이 시각화
#
# 여러 캡처를 겹쳐서 보면 AES 라운드 구조가 더 명확하게 보입니다.
# ─────────────────────────────────────────────────────────────

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span
from bokeh.resources import INLINE
from bokeh.palettes import Category10

output_notebook(INLINE)

p2 = figure(
    width=1200,
    height=450,
    title=f"다중 캡처 오버레이 ({min(5, len(valid_indices))}개)",
    x_axis_label="ADC 샘플",
    y_axis_label="전력",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

colors = Category10[10]
display_count = min(5, len(valid_indices))

for idx, vi in enumerate(valid_indices[:display_count]):
    pt = powertraces_list[vi]
    tt = trace_times_list[vi]

    # 전력 파형 (다른 색상으로)
    xrange = list(range(len(pt.wave)))
    p2.line(
        xrange, pt.wave,
        line_color=colors[idx],
        line_width=1,
        alpha=0.6,
        legend_label=f"파형 {vi+1}"
    )

# 첫 번째 유효 캡처의 트레이스 이벤트만 표시
if valid_indices:
    for t in trace_times_list[valid_indices[0]]:
        p2.renderers.append(
            Span(location=t[0]*multiplier, dimension='height',
                 line_color='black', line_width=1, line_alpha=0.5)
        )

p2.legend.location = "top_right"
p2.legend.click_policy = "hide"  # 범례 클릭으로 파형 표시/숨기기 가능
show(p2)

print("\n💡 팁: 범례를 클릭하면 개별 파형을 숨기거나 표시할 수 있습니다.")

---
## Step 10: 트레이스 기반 정밀 트리거

TraceWhisperer의 강력한 활용 방법 중 하나는  
트레이스 이벤트를 **전력 캡처의 트리거**로 사용하는 것입니다.

일반적인 트리거 방식:
- 펌웨어 트리거: AES 연산 전체를 포함하는 긴 구간 캡처
- 트레이스 트리거: 특정 함수 호출 시점 바로 전후만 캡처 → ADC 샘플 절약

In [ ]:
# ─────────────────────────────────────────────────────────────
# 분석: 이벤트 간 시간 간격 계산
#
# AES-128의 라운드 구조를 파형에서 확인합니다.
# SubBytes → AddRoundKey → SubBytes → AddRoundKey ...
# ─────────────────────────────────────────────────────────────

if valid_indices:
    sample_times = trace_times_list[valid_indices[0]]

    print("=" * 60)
    print("AES 라운드별 이벤트 분석 (첫 번째 유효 캡처)")
    print("=" * 60)
    print(f"{'이벤트':>6}  {'타임스탬프':>12}  {'간격(사이클)':>12}")
    print("-" * 40)

    for i, t in enumerate(sample_times):
        if i == 0:
            interval = "-"
        else:
            interval = f"{t[0] - sample_times[i-1][0]:>12,}"
        print(f"{i+1:>6}  {t[0]:>12,}  {interval}")

    # 평균 라운드 주기 계산
    if len(sample_times) >= 4:
        intervals = [sample_times[i][0] - sample_times[i-1][0]
                     for i in range(1, len(sample_times))]
        print(f"\n평균 이벤트 간격: {np.mean(intervals):,.0f} 사이클")
        print(f"(≈ {np.mean(intervals) / scope.clock.clkgen_freq * 1e6:.2f} μs @ {scope.clock.clkgen_freq/1e6:.2f} MHz)")

---
## Step 11: 정리 및 연결 해제

In [ ]:
# ─────────────────────────────────────────────────────────────
# 장비 연결 정리
#
# 노트북 사용이 끝나면 반드시 연결을 해제해야 합니다.
# 그렇지 않으면 다음 세션에서 연결 오류가 발생할 수 있습니다.
# ─────────────────────────────────────────────────────────────

scope.dis()
target.dis()
print("✅ 장비 연결 해제 완료")
print("   다음 사용 시 Step 2부터 다시 실행하세요.")

In [ ]:
isClean = True

if isClean:
    !cd simpleserial-trace && make PLATFORM=$PLATFORM clean && rm -f simpleserial-trace-CW308_STM32F3.txt

---
## 정리: 배운 내용 요약

이 튜토리얼에서 학습한 내용:

### TraceWhisperer 설정 순서

```
1. scope.trace.enabled = True          → TraceWhisperer 활성화
2. scope.trace.trace_mode = 'SWO'      → SWO 인터페이스 선택
3. scope.trace.jtag_to_swd()           → JTAG → SWD 전환
4. SWO 클럭 파라미터 설정              → ACPR, swo_div
5. DWT_CTRL 설정                       → 싱크 프레임 비활성화
6. capture.trigger_source 설정         → 트리거 소스 선택
7. set_pattern_match() + rules_enabled → 패턴 매칭 설정
8. capture.mode 설정                   → 캡처 지속 조건
9. set_isync_matches()                 → DWT_COMP 주소 등록
10. arm_trace() → capture_trace()      → 실제 캡처
11. read_capture_data()                → 데이터 읽기
12. get_rule_match_times()             → 타임스탬프 파싱
```

### 핵심 개념

- **DWT_COMP**: 함수 시작 주소를 등록하면 PC가 해당 주소를 통과할 때 이벤트 발생
- **패턴 매칭**: FPGA 내부에서 SWO 스트림을 분석, 소프트웨어 파싱 불필요
- **타임스탬프 스케일**: 트레이스 타임스탬프 × `adc_mul` = 파형 x축 좌표

### 다음 단계

1. **pc_sample_annotate.ipynb**: 함수 실행 구간을 파형에 색상으로 표시
2. **uecc.ipynb**: 트레이스로 ECC 소프트웨어 취약점 공격 보조
3. **trace_clock_alignment.ipynb**: 병렬 트레이스의 클럭 위상 조정

---

> **💡 실습 과제**: PC 주소를 바꿔가며 AES의 다른 함수들 (`ShiftRows`, `MixColumns`)의  
> 실행 시점을 파형에서 찾아보세요!

---
## 부록: 자주 발생하는 오류와 해결 방법

### ❌ `AssertionError: Hmm, the clock you chose doesn't seem to be active`
- **원인**: 타겟 클럭이 Husky에 도달하지 않음
- **해결**: `scope.io.hs2 = 'clkgen'` 설정 확인, 20핀 커넥터 연결 확인

### ❌ `AssertionError: SWO line not high`
- **원인**: TDO → D2 점퍼 케이블 미연결 또는 SWD 전환 실패
- **해결**: 케이블 연결 확인, `reset_target()` 후 `jtag_to_swd()` 재실행

### ❌ `trace.fifo_empty() == True` (트레이스 이벤트 없음)
- **원인**: PC 주소 불일치, 패턴 매칭 미설정, 펌웨어 불일치
- **해결**: `simpleserial-trace` 펌웨어 확인, PC 주소 재확인 (objdump로 검증)

### ❌ 이벤트 수가 21개가 아닌 경우
- **원인**: SWO 대역폭 부족 (이벤트 손실), 또는 다른 펌웨어 사용
- **해결**: `ACPR` 값을 높여 SWO 속도를 낮추거나, 두 주소 중 하나만 모니터링

### ❌ `powertrace is None`
- **원인**: 타겟이 정해진 시간 내에 응답하지 않음
- **해결**: `reset_target(scope)` 후 재시도, `scope.adc.timeout` 값 증가